# Step 0 — Sector definition and validation

## Scope
Precision agriculture applied to crop production.
Livestock and aquaculture are outside our scope.

## Initial search strategy
The candidate search is `CPC OR [core keywords OR (agricultural terms AND technological terms)]`. The lists and analysis period are read from config/sector.yaml

This is a WIPO-inspired adaptation, not a literal replication: this notebook searches titles and abstracts and uses CPC classifications.

## 1. Load the search configuration

We read the sector definition, CPC prefixes and keyword lists from
`config/sector.yaml`.

In [1]:
from pathlib import Path
import yaml
import re
import pandas as pd
import numpy as np

# Locate the repository from either its root or notebooks folder.
current_dir = Path.cwd()

if (current_dir / "config" / "sector.yaml").is_file():
    repo_dir = current_dir
elif (current_dir.parent / "config" / "sector.yaml").is_file():
    repo_dir = current_dir.parent
else:
    raise FileNotFoundError("Cannot find config/sector.yaml")

config_path = repo_dir / "config" / "sector.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

search = config["search"]

## 2. Connect to PATSTAT

In [4]:
from sqlalchemy import and_, or_, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

# Use TEST to check that the code works.
# Switch to PROD before drawing the actual validation sample.
#ENV = "TEST"
ENV = "PROD"

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print("PATSTAT environment:", ENV)

PATSTAT environment: PROD


### 3. Define the search conditions
- **CPC route**: at least one classification matches a configured prefix.
- **Keyword route**: Either a SPECIFIC precision agriculture keyword OR at least one agricultural term AND at least one technological term occur in the same application's title and/or abstract. 
$$\text{Keywords} =\text{core keywords} \cup (\text{Agricultural terms} \cap \text{Technological terms)}$$
A CPC match does not require a keyword match.

In [5]:
import re
from sqlalchemy import and_, or_, func, false

search = config["search"]

AGRICULTURAL_TERMS = search["agricultural_terms"]
TECHNOLOGICAL_TERMS = search["technological_terms"]
SPECIFIC_PHRASES = search["specific_phrases"]
SECTOR_CPC_PREFIXES = search["cpc_prefixes"]
REVIEW_EXCLUSION_TERMS = search.get("review_exclusion_terms", [])

# Only these terms allow word endings; other terms match whole words.
STEM_TERMS = {
    "agricultur", "cultivat", "horticultur",
    "crop", "drone", "robot", "auto-steer",
}


def keyword_condition(terms, allow_stems=True):
    """Match alternatives in the title or abstract."""
    title = func.lower(
        func.coalesce(TLS202_APPLN_TITLE.appln_title, "")
    )
    abstract = func.lower(
        func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, "")
    )

    patterns = []

    for term in terms:
        term = term.strip().lower()
        if not term:
            raise ValueError("Search terms must not be empty.")

        # Accept spaces or hyphens between words.
        words = re.split(r"[\s-]+", term)
        phrase = r"[\s-]+".join(re.escape(word) for word in words)

        suffix = (
            r"\w*"
            if allow_stems and term in STEM_TERMS
            else ""
        )
        patterns.append(r"\b" + phrase + suffix + r"\b")

    if not patterns:
        return false()

    pattern = "(?:" + "|".join(patterns) + ")"

    return or_(
        func.regexp_contains(title, pattern),
        func.regexp_contains(abstract, pattern),
    )


def symbol_prefix_condition(symbol_column, codes):
    """Match CPC prefixes after removing spaces."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(
        symbol.like(code.replace(" ", "") + "%")
        for code in codes
    ))


def query_by_keywords(*columns):
    """Specific phrases OR (agricultural AND technological terms)."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(
            TLS202_APPLN_TITLE,
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        )
        .filter(
            or_(
                keyword_condition(
                    SPECIFIC_PHRASES, allow_stems=False
                ),
                and_(
                    keyword_condition(AGRICULTURAL_TERMS),
                    keyword_condition(TECHNOLOGICAL_TERMS),
                ),
            )
        )
    )


def query_by_cpc(*columns):
    """Select applications matching the configured CPC prefixes."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(
            TLS224_APPLN_CPC,
            TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id,
        )
        .filter(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SECTOR_CPC_PREFIXES,
            )
        )
    )


print("Updated search functions are ready.")

Updated search functions are ready.


## 4. Combine the routes at family level
Keep patent-invention applications (`ipr_type == "PI"`) with a positive DOCDB family ID. UNION combines both routes and removes duplicate family IDs. A family enters if at least one member application qualifies.
$$\text{Precision Agriculture} = \text{Keywords} \cup \text{CPC}$$

In [9]:
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

base_filters = (
    TLS201_APPLN.docdb_family_id > 0,
    TLS201_APPLN.ipr_type == "PI",
)

families_by_cpc = (
    query_by_cpc(family_id)
    .filter(*base_filters)
)

families_by_keywords = (
    query_by_keywords(family_id)
    .filter(*base_filters)
)

# CPC OR keywords; UNION removes duplicate family IDs.
sector_ids = families_by_cpc.union(families_by_keywords)

## 5. Apply the analysis period and count candidate families
For each selected family, calculate the minimum of the configured time field across its patent-invention applications. Apply the inclusive year range from the YAML afterwards.

The interpretation of the date depends on `family_time_column`: do not label a filing-year variable as a directly observed priority date without checking its definition. The result is a count of families, not individual applications.

In [10]:
YEAR_MIN = config["analysis"]["year_min"]
YEAR_MAX = config["analysis"]["year_max"]
FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

# Use the earliest available filing year across the family's
# patent-invention applications.
family_year = func.min(
    getattr(TLS201_APPLN, FAMILY_TIME_COLUMN)
)

sector_families_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
        family_year.label("family_year"),
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(
            sector_ids.subquery().select()
        ),
        TLS201_APPLN.ipr_type == "PI",
    )
    .group_by(TLS201_APPLN.docdb_family_id)

    # Apply the time window after calculating the family-level year.
    .having(family_year.between(YEAR_MIN, YEAR_MAX))
)

preview = patstat.df(
    sector_families_query
    .order_by(TLS201_APPLN.docdb_family_id)
    .limit(10)
)

preview

,docdb_family_id,family_year
0,3828303,2001
1,3828568,2001
2,7628526,2000
3,7639055,2000
4,7643476,2000
5,7646835,2000
6,7662531,2000
7,7668867,2000
8,7669955,2001
9,7682617,2001


In [11]:
# Count the selected families without downloading the full dataset.
family_scope = sector_families_query.subquery()

count_query = (
    db.query(func.count().label("n_families"))
    .select_from(family_scope)
)

family_count = patstat.df(count_query)
family_count

if int(family_count.iloc[0]["n_families"]) == 0:
    raise ValueError("No candidate families. Check the configuration before continuing.")
display(family_count)

,n_families
0,10373


## 6. Measure the contribution of each search route
Count mutually exclusive groups within the same candidate population and time window: CPC only, keywords only, and both. Their sum must equal the total family count. These counts describe retrieval, not relevance.

In [12]:
# Use the same family population and time window as the total count.
scope = sector_families_query.subquery()
cpc_ids = families_by_cpc.subquery()
keyword_ids = families_by_keywords.subquery()

has_cpc = scope.c.docdb_family_id.in_(
    db.query(cpc_ids.c.docdb_family_id)
)
has_keywords = scope.c.docdb_family_id.in_(
    db.query(keyword_ids.c.docdb_family_id)
)

# Each family belongs to exactly one group.
groups = {
    "CPC only": and_(has_cpc, ~has_keywords),
    "Keywords only": and_(has_keywords, ~has_cpc),
    "Both CPC and keywords": and_(has_cpc, has_keywords),
}

rows = []

for group, condition in groups.items():
    query = (
        db.query(func.count().label("n_families"))
        .select_from(scope)
        .filter(condition)
    )

    count = int(patstat.df(query).iloc[0]["n_families"])
    rows.append({"group": group, "n_families": count})

branch_counts = pd.DataFrame(rows)

# Check that the groups account for the entire candidate dataset.
total = int(family_count.iloc[0]["n_families"])
assert branch_counts["n_families"].sum() == total, "Counts do not match."

branch_counts["share_pct"] = (
    100 * branch_counts["n_families"] / total
).round(1)

branch_counts

,group,n_families,share_pct
0,CPC only,1982,19.1
1,Keywords only,8096,78.0
2,Both CPC and keywords,295,2.8


## Understanding the keyword-only families

We examine families retrieved through keywords but not through the CPC route.

We distinguish three mutually exclusive groups:
- specific phrases only;
- agricultural AND technological terms only;
- both keyword routes.

This shows which part of the text search contributes additional families.
It does not establish whether those families are relevant.

In [14]:
# Reuse the title/abstract joins for both text-search routes.
def query_by_text_condition(condition):
    return (
        db.query(
            TLS201_APPLN.docdb_family_id.label("docdb_family_id")
        )
        .select_from(TLS201_APPLN)
        .outerjoin(
            TLS202_APPLN_TITLE,
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        )
        .filter(
            TLS201_APPLN.docdb_family_id > 0,
            TLS201_APPLN.ipr_type == "PI",
            condition,
        )
        .distinct()
    )


phrase_ids = query_by_text_condition(
    keyword_condition(SPECIFIC_PHRASES, allow_stems=False)
).subquery()

agri_tech_ids = query_by_text_condition(
    and_(
        keyword_condition(AGRICULTURAL_TERMS),
        keyword_condition(TECHNOLOGICAL_TERMS),
    )
).subquery()

cpc_ids_check = families_by_cpc.distinct().subquery()
text_scope = sector_families_query.subquery()

matches_phrase = text_scope.c.docdb_family_id.in_(
    db.query(phrase_ids.c.docdb_family_id)
)

matches_agri_tech = text_scope.c.docdb_family_id.in_(
    db.query(agri_tech_ids.c.docdb_family_id)
)

matches_cpc = text_scope.c.docdb_family_id.in_(
    db.query(cpc_ids_check.c.docdb_family_id)
)

text_groups = {
    "Specific phrases only": and_(
        matches_phrase, ~matches_agri_tech
    ),
    "Agricultural AND technological only": and_(
        matches_agri_tech, ~matches_phrase
    ),
    "Both text routes": and_(
        matches_phrase, matches_agri_tech
    ),
}

rows = []

for group, condition in text_groups.items():
    query = (
        db.query(func.count().label("n_families"))
        .select_from(text_scope)
        .filter(~matches_cpc, condition)
    )

    count = int(patstat.df(query).iloc[0]["n_families"])
    rows.append({"group": group, "n_families": count})

keyword_breakdown = pd.DataFrame(rows)

expected_total = int(
    branch_counts.loc[
        branch_counts["group"] == "Keywords only",
        "n_families",
    ].iloc[0]
)

assert keyword_breakdown["n_families"].sum() == expected_total, (
    "Counts differ: rerun the earlier search and counting cells."
)

keyword_breakdown["share_pct"] = (
    (100 * keyword_breakdown["n_families"] / expected_total).round(1)
    if expected_total > 0
    else 0.0
)

print("Families retrieved through keywords without a CPC match:",
      expected_total)

keyword_breakdown

Families retrieved through keywords without a CPC match: 8096


,group,n_families,share_pct
0,Specific phrases only,219,2.7
1,Agricultural AND technological only,7785,96.2
2,Both text routes,92,1.1


In [17]:
# Retrieve family IDs from the broad text-only group.
broad_only_query = (
    db.query(text_scope.c.docdb_family_id)
    .select_from(text_scope)
    .filter(
        ~matches_cpc,
        ~matches_phrase,
        matches_agri_tech,
    )
)

broad_only_data = patstat.df(broad_only_query)

# Reproducible exploratory sample.
broad_sample = (
    broad_only_data
    .sort_values("docdb_family_id")
    .sample(n=min(20, len(broad_only_data)), random_state=38)
    .reset_index(drop=True)
)

broad_sample

,docdb_family_id
0,82769582
1,71191042
2,82215118
3,59138939
4,53333931
5,79177730
6,70316674
7,65963460
8,55140619
9,60986526


## 7. Semantic screening with Patent SBERT
Compare one English abstract per family with a description of the intended scope. Families without an English abstract remain in the candidate dataset but cannot be scored here.

If sentence_transformers is not installed, run %pip install -q sentence-transformers in a separate setup cell, then restart the kernel and rerun the notebook. Model loading may download model files.
### 7.1 Select a representative English abstract
Choose the earliest-filed application with a non-empty English abstract, breaking ties by application ID and abstract text. This is reproducible, but one abstract may not describe every aspect of a family.

In [13]:
# Select the families already included in our analysis.
scope_for_sbert = sector_families_query.subquery()

# Rank applications within each family, keeping English abstracts only.
ranked_abstracts = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
        TLS201_APPLN.appln_id.label("appln_id"),
        TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
        func.row_number().over(
            partition_by=TLS201_APPLN.docdb_family_id,
            order_by=(
                TLS201_APPLN.appln_filing_date,
                TLS201_APPLN.appln_id,
                TLS203_APPLN_ABSTR.appln_abstract,
            ),
        ).label("row_number"),
    )
    .select_from(TLS201_APPLN)
    .join(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(
            db.query(scope_for_sbert.c.docdb_family_id)
        ),
        TLS201_APPLN.ipr_type == "PI",
        func.lower(
            func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)
        ) == "en",
        func.length(
            func.trim(TLS203_APPLN_ABSTR.appln_abstract)
        ) > 0,
    )
    .subquery()
)

# Download one representative abstract per family.
abstract_query = (
    db.query(
        ranked_abstracts.c.docdb_family_id,
        ranked_abstracts.c.appln_id,
        ranked_abstracts.c.abstract,
    )
    .filter(ranked_abstracts.c.row_number == 1)
    .order_by(ranked_abstracts.c.docdb_family_id)
)

sbert_data = patstat.df(abstract_query)

total_families = int(family_count.iloc[0]["n_families"])

print("Candidate families:", total_families)
print("Families with an English abstract:", len(sbert_data))
print("Families not scored:", total_families - len(sbert_data))

if sbert_data.empty:
    raise ValueError("No English abstracts available for scoring.")

Candidate families: 10373
Families with an English abstract: 10152
Families not scored: 221


### 7.2 Load the model and define the scope description
Keep the model and description unchanged when comparing search variants. Text longer than the model's token limit may be truncated.

In [ ]:
from sentence_transformers import SentenceTransformer

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"

PA_DESCRIPTION = """
Systems, devices and methods for precision agriculture in crop production,
measuring spatial or temporal variation in soil and crop conditions and
supporting site-specific management and targeted agricultural operations.
Inventions may address one or more functions: soil and crop sensing,
remote sensing, yield monitoring and mapping, geospatial data processing,
georeferenced field mapping and prescription maps, GNSS-based guidance
or automatic steering of agricultural machinery, variable-rate application
of seed, fertilizer, irrigation water or crop protection products, and
selective detection and treatment of individual plants or weeds.
""".strip()

model = SentenceTransformer(SBERT_MODEL)

print("Model:", SBERT_MODEL)
print("Maximum input length:", model.max_seq_length, "tokens")

### 7.3 Calculate cosine similarity
Higher scores indicate greater semantic similarity to the description. 

In [ ]:
# Convert each abstract into a normalised numerical representation.
abstract_embeddings = model.encode(
    sbert_data["abstract"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Encode the scope description using the same model.
description_embedding = model.encode(
    PA_DESCRIPTION,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# For normalised vectors, the dot product equals cosine similarity.
sbert_data["cosine_similarity"] = (
    abstract_embeddings @ description_embedding
)

sbert_ranked = (
    sbert_data.sort_values(
        ["cosine_similarity", "docdb_family_id"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

sbert_ranked.insert(0, "rank", range(1, len(sbert_ranked) + 1))

sbert_ranked[
    ["rank", "docdb_family_id", "appln_id", "cosine_similarity"]
].head(10)

### 7.4 Describe the score distribution
The percentages below refer only to scored families. The intervals are descriptive bins.

In [ ]:
scores = sbert_data["cosine_similarity"].dropna()

# Descriptive ranges, not validated relevance thresholds.
bands = pd.cut(
    scores,
    bins=[-np.inf, 0, 0.2, 0.4, 0.6, 0.8, np.inf],
    labels=[
        "< 0",
        "0–<0.2",
        "0.2–<0.4",
        "0.4–<0.6",
        "0.6–<0.8",
        "≥ 0.8",
    ],
    right=False,
)

similarity_counts = (
    bands.value_counts(sort=False)
    .rename_axis("cosine_similarity")
    .reset_index(name="n_families")
)

similarity_counts["share_pct"] = (
    100 * similarity_counts["n_families"] / len(scores)
).round(1)

print("Total families scored:", len(scores))
similarity_counts

### 7.5 UMAP
Project the SBERT embeddings into two dimensions using the same UMAP
parameters as the course tutorial.

Each point represents one family's selected English abstract.
The Precision Agriculture description is projected together with the
abstracts and displayed as a star. Colours show cosine similarity
calculated in the original embedding space.

The map supports exploration: distances in two dimensions are approximate
and do not provide a validated criterion for excluding families.

In [ ]:
%pip install -q umap-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

# Keep the original row order: embeddings were created from sbert_data.
umap_data = sbert_data.copy()

assert len(umap_data) == len(abstract_embeddings), (
    "Rerun the SBERT cells: rows and embeddings do not match."
)

# Project the abstracts and sector description together.
umap_input = np.vstack([
    abstract_embeddings,
    description_embedding.reshape(1, -1),
])

# Same parameters as the professor's notebook.
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)

coords = reducer.fit_transform(umap_input)

umap_data[["UMAP_1", "UMAP_2"]] = coords[:-1]
pa_xy = coords[-1]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

points = ax.scatter(
    umap_data["UMAP_1"],
    umap_data["UMAP_2"],
    c=umap_data["cosine_similarity"],
    cmap="viridis",
    s=18,
    alpha=0.75,
)

ax.scatter(
    pa_xy[0],
    pa_xy[1],
    marker="*",
    s=400,
    color="red",
    edgecolors="black",
    linewidths=0.8,
    label="Precision Agriculture description",
    zorder=5,
)

fig.colorbar(
    points,
    ax=ax,
    label="Cosine similarity to the sector description",
)

ax.set(
    xlabel="UMAP dimension 1",
    ylabel="UMAP dimension 2",
    title="Precision Agriculture — semantic map of patent families",
)

ax.legend()
plt.tight_layout()
plt.show()

## 8. Inspect examples
Use the same display function for low or high scores. Change START to read the next batch.

A score below 0.2 is only a diagnostic flag. Read the text before judging relevance; missing context, terminology and truncation can affect scores. Low-scoring examples are not a representative random validation sample.

In [ ]:
LOW_SCORE_THRESHOLD = 0.2
low_similarity_review = (
    sbert_ranked.loc[sbert_ranked["cosine_similarity"] < LOW_SCORE_THRESHOLD]
    .sort_values(["cosine_similarity", "docdb_family_id"])
    .reset_index(drop=True)
    .copy()
)
print("Low-scoring families:", len(low_similarity_review))


def show_examples(data, start=0, size=5):
    """Display a batch without changing the dataset or assigning labels."""
    for index, row in data.iloc[start:start + size].iterrows():
        print(f"\nROW: {index}")
        print(f"FAMILY: {row['docdb_family_id']}")
        print(f"PATSTAT APPLICATION ID: {row['appln_id']}")
        print(f"COSINE SIMILARITY: {row['cosine_similarity']:.3f}")
        print("ABSTRACT:", row["abstract"])
        print("-" * 80)


# Use sbert_ranked to inspect the highest scores instead.
START = 0
show_examples(low_similarity_review, start=START, size=5)

### 8.1 Check how the inspected families were retrieved
The table below checks the first 15 low-scoring families against both original search routes. True indicates a retrieval match, not a human judgment of relevance. Keyword matches may come from another application in the family, rather than the abstract displayed above.

In [ ]:
# Inspect the retrieval route of the first 15 low-scoring families.
diagnostic_ids = (
    low_similarity_review.iloc[:15]["docdb_family_id"]
    .astype(int)
    .tolist()
)

cpc_matches = patstat.df(
    families_by_cpc
    .filter(TLS201_APPLN.docdb_family_id.in_(diagnostic_ids))
    .distinct()
)

keyword_matches = patstat.df(
    families_by_keywords
    .filter(TLS201_APPLN.docdb_family_id.in_(diagnostic_ids))
    .distinct()
)

# An empty query result may have no columns.
cpc_family_ids = (
    set() if cpc_matches.empty
    else set(cpc_matches["docdb_family_id"])
)

keyword_family_ids = (
    set() if keyword_matches.empty
    else set(keyword_matches["docdb_family_id"])
)

print("Diagnostic families matching CPC:", len(cpc_family_ids))
print("Diagnostic families matching keywords:", len(keyword_family_ids))

retrieval_check = low_similarity_review.iloc[:15][
    ["docdb_family_id", "cosine_similarity"]
].copy()

retrieval_check["matched_cpc"] = (
    retrieval_check["docdb_family_id"].isin(cpc_family_ids)
)
retrieval_check["matched_keywords"] = (
    retrieval_check["docdb_family_id"].isin(keyword_family_ids)
)

retrieval_check